In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import numpy as np

from utils import load_dataset, CosmicWebDataset
from transform import SimpleResize
from models import WDMClassifierLarge

In [2]:
# Evaluation Function
def evaluate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            labels = labels.to(device).unsqueeze(1)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            total_loss += loss.item()
            preds = (torch.sigmoid(outputs) > 0.5).float()
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    
    avg_loss = total_loss / len(dataloader)
    accuracy = correct / total
    return avg_loss, accuracy

# Main Training Function
def train_cnn(cdm_file, wdm_file):
    # Configuration
    config = {
        'batch_size': 32,
        'lr': 2e-4,
        'epochs': 20,
        'dropout': 0.1,
        'device': 'cuda' if torch.cuda.is_available() else 'cpu'
    }
    
    print("Loading data...")
   
    # Create sample indices
    n_samples = 1000  # Adjust based on your data size
    indices = list(range(n_samples))
    np.random.shuffle(indices)
    
    # Create train/test split
    split_idx = int(0.8 * len(indices))
    train_indices = indices[:split_idx]
    test_indices = indices[split_idx:]
    
    print(f"Train samples: {len(train_indices)}, Test samples: {len(test_indices)}")

    cdm = np.log1p(np.load(cdm_file))
    wdm = np.log1p(np.load(wdm_file))
    total = np.concatenate((cdm,wdm))
    mean, std = total.mean(), total.std()
    stats = {
        'mean': mean,
        'std': std
    }
    
    rescale = SimpleResize(
        size=(256,256),
        apply_log=True,
        normalize=stats
    )
    
    train_dataset = load_dataset(train_indices, transform=rescale, 
                               cdm_file=cdm_file, wdm_file=wdm_file)
    test_dataset = load_dataset(test_indices, transform=rescale, 
                              cdm_file=cdm_file, wdm_file=wdm_file)
    
    train_loader = DataLoader(train_dataset, batch_size=config['batch_size'], shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=config['batch_size'], shuffle=False)
    
    # Initialize model, loss, optimizer
    device = torch.device(config['device'])
    model = WDMClassifierLarge(dropout=config['dropout']).to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.AdamW(model.parameters(), lr=config['lr'])
    
    print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
    print(f"Using device: {device}")
    
    # Training loop
    train_losses = []
    test_losses = []
    test_accuracies = []
    
    print(f"\nStarting training for {config['epochs']} epochs...")
    print("=" * 70)
    
    for epoch in range(config['epochs']):
        # Training phase
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0
        
        for batch_idx, (images, labels) in enumerate(train_loader):
            images = images.to(device)
            labels = labels.to(device).unsqueeze(1)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            preds = (torch.sigmoid(outputs) > 0.5).float()
            train_correct += (preds == labels).sum().item()
            train_total += labels.size(0)
        
        # Calculate averages
        avg_train_loss = train_loss / len(train_loader)
        train_acc = train_correct / train_total
        
        # Test evaluation
        test_loss, test_acc = evaluate(model, test_loader, criterion, device)
        
        # Store metrics
        train_losses.append(avg_train_loss)
        test_losses.append(test_loss)
        test_accuracies.append(test_acc)
        
        # Print progress
        print(f"Epoch {epoch+1:2d}/{config['epochs']:2d} | "
              f"Train Loss: {avg_train_loss:.4f} | "
              f"Train Acc: {train_acc:.4f} | "
              f"Test Loss: {test_loss:.4f} | "
              f"Test Acc: {test_acc:.4f}")
    
    print("=" * 70)
    print("Training completed!")
    
    print(f"\nFinal Results:")
    print(f"Final Test Loss: {test_losses[-1]:.4f}")
    print(f"Final Test Accuracy: {test_accuracies[-1]:.4f}")
    
    return model, train_losses, test_losses, test_accuracies


In [3]:
cdm_file='/n/netscratch/iaifi_lab/Lab/msliu/CMD/data/IllustrisTNG/Maps_Mcdm_IllustrisTNG_LH_z=0.00.npy'
wdm_file='/n/netscratch/iaifi_lab/Lab/ccuestalazaro/DREAMS/Images/WDM/boxes/Maps_Mcdm_IllustrisTNG_WDM_z=0.00.npy'

model, train_losses, test_losses, test_accuracies = train_cnn(cdm_file, wdm_file)

Loading data...
Train samples: 800, Test samples: 200
Loading CDM data from /n/netscratch/iaifi_lab/Lab/msliu/CMD/data/IllustrisTNG/Maps_Mtot_IllustrisTNG_LH_z=0.00.npy...
CDM data shape: (15000, 256, 256)
Loading WDM data from /n/netscratch/iaifi_lab/Lab/ccuestalazaro/DREAMS/Images/WDM/boxes/Maps_Mtot_IllustrisTNG_WDM_z=0.00.npy...
WDM data shape: (15360, 256, 256)
Created dataset with 1600 samples
Loading CDM data from /n/netscratch/iaifi_lab/Lab/msliu/CMD/data/IllustrisTNG/Maps_Mtot_IllustrisTNG_LH_z=0.00.npy...
CDM data shape: (15000, 256, 256)
Loading WDM data from /n/netscratch/iaifi_lab/Lab/ccuestalazaro/DREAMS/Images/WDM/boxes/Maps_Mtot_IllustrisTNG_WDM_z=0.00.npy...
WDM data shape: (15360, 256, 256)
Created dataset with 400 samples
Model parameters: 4,503,681
Using device: cuda

Starting training for 20 epochs...
Epoch  1/20 | Train Loss: 0.6982 | Train Acc: 0.4963 | Test Loss: 0.7006 | Test Acc: 0.5000
Epoch  2/20 | Train Loss: 0.6964 | Train Acc: 0.4956 | Test Loss: 0.6973 |

KeyboardInterrupt: 

In [4]:
cdm_file='/n/netscratch/iaifi_lab/Lab/msliu/CMD/data/IllustrisTNG/Maps_Mcdm_IllustrisTNG_LH_z=0.00.npy'
wdm_file='/n/netscratch/iaifi_lab/Lab/msliu/Maps_Mcdm_IllustrisTNG_WDM_z=0.00.npy'

model, train_losses, test_losses, test_accuracies = train_cnn(cdm_file, wdm_file)

Loading data...
Train samples: 800, Test samples: 200
Loading CDM data from /n/netscratch/iaifi_lab/Lab/msliu/CMD/data/IllustrisTNG/Maps_Mcdm_IllustrisTNG_LH_z=0.00.npy...
CDM data shape: (15000, 256, 256)
Loading WDM data from /n/netscratch/iaifi_lab/Lab/msliu/Maps_Mcdm_IllustrisTNG_WDM_z=0.00.npy...
WDM data shape: (2000, 256, 256)
Created dataset with 1600 samples
Loading CDM data from /n/netscratch/iaifi_lab/Lab/msliu/CMD/data/IllustrisTNG/Maps_Mcdm_IllustrisTNG_LH_z=0.00.npy...
CDM data shape: (15000, 256, 256)
Loading WDM data from /n/netscratch/iaifi_lab/Lab/msliu/Maps_Mcdm_IllustrisTNG_WDM_z=0.00.npy...
WDM data shape: (2000, 256, 256)
Created dataset with 400 samples
Model parameters: 4,503,681
Using device: cuda

Starting training for 20 epochs...
Epoch  1/20 | Train Loss: 0.7026 | Train Acc: 0.4894 | Test Loss: 0.7114 | Test Acc: 0.5000
Epoch  2/20 | Train Loss: 0.6933 | Train Acc: 0.5000 | Test Loss: 0.7194 | Test Acc: 0.4725
Epoch  3/20 | Train Loss: 0.6910 | Train Acc: 0.